<a href="https://colab.research.google.com/github/MulukenChernetET/OIBSIP/blob/main/DataAnalytics-L1-EDARetailSales/Cleaning__Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Customer Call List.xlsx to Customer Call List.xlsx


In [ ]:
import pandas as pd
import numpy as np

# Load the messy Customer Call List dataset
df = pd.read_excel('Customer Call List.xlsx')

# First look
df.head(10)


,CustomerID,First_Name,Last_Name,Phone_Number,Address,Paying Customer,Do_Not_Contact,Not_Useful_Column
0,1001,Frodo,Baggins,123-545-5421,"123 Shire Lane, Shire",Yes,No,True
1,1002,Abed,Nadir,123/643/9775,93 West Main Street,No,Yes,False
2,1003,Walter,/White,7066950392,298 Drugs Driveway,N,NaN,True
3,1004,Dwight,Schrute,123-543-2345,"980 Paper Avenue, Pennsylvania, 18503",Yes,Y,True
4,1005,Jon,Snow,876|678|3469,123 Dragons Road,Y,No,True
5,1006,Ron,Swanson,304-762-2467,768 City Parkway,Yes,Yes,True
6,1007,Jeff,Winger,NaN,1209 South Street,No,No,False
7,1008,Sherlock,Holmes,876|678|3469,98 Clue Drive,N,No,False
8,1009,Gandalf,NaN,N/a,123 Middle Earth,Yes,NaN,False
9,1010,Peter,Parker,123-545-5421,"25th Main Street, New York",Yes,No,True


In [ ]:
# Basic info
print("Shape (rows, columns):", df.shape)
print("\nColumn info:")
df.info()

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())

Shape (rows, columns): (21, 8)

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   CustomerID         21 non-null     int64 
 1   First_Name         21 non-null     object
 2   Last_Name          20 non-null     object
 3   Phone_Number       19 non-null     object
 4   Address            21 non-null     object
 5   Paying Customer    21 non-null     object
 6   Do_Not_Contact     17 non-null     object
 7   Not_Useful_Column  21 non-null     bool  
dtypes: bool(1), int64(1), object(6)
memory usage: 1.3+ KB

Missing values per column:
CustomerID           0
First_Name           0
Last_Name            1
Phone_Number         2
Address              0
Paying Customer      0
Do_Not_Contact       4
Not_Useful_Column    0
dtype: int64

Number of duplicate rows: 1


In [ ]:
# Inspect categorical columns for inconsistent formatting
print(df['Paying Customer'].unique())
print(df['Do_Not_Contact'].unique())
print(df['Phone_Number'].unique())

['Yes' 'No' 'N' 'Y' 'N/a']
['No' 'Yes' nan 'Y' 'N']
['123-545-5421' '123/643/9775' 7066950392 '123-543-2345' '876|678|3469'
 '304-762-2467' nan 'N/a']


In [ ]:
# Standardize Yes/No style columns to a consistent format
df['Paying Customer'] = df['Paying Customer'].str.strip().str.upper().replace({'YES': 'Y', 'NO': 'N'})
df['Do_Not_Contact'] = df['Do_Not_Contact'].str.strip().str.upper().replace({'YES': 'Y', 'NO': 'N'})

In [ ]:
# Last_Name (1 missing): can't infer a name — flag as 'Unknown' rather than drop the row
df['Last_Name'] = df['Last_Name'].fillna('Unknown')

# Phone_Number (2 missing): no reliable way to impute a phone number — leave as null,
# but flag with a boolean column so it's traceable in analysis
df['Phone_Missing'] = df['Phone_Number'].isnull()

# Do_Not_Contact (4 missing): treat missing as 'N' (assume contactable unless explicitly opted out)
df['Do_Not_Contact'] = df['Do_Not_Contact'].fillna('N')

In [ ]:
dupe_count_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {dupe_count_before} duplicate row(s)")

Removed 1 duplicate row(s)


In [ ]:
# Not_Useful_Column adds no analytical value per its own label — drop it
df = df.drop(columns=['Not_Useful_Column'])

In [ ]:
df['CustomerID'] = df['CustomerID'].astype(str) # ID should be treated as identifier, not numeric
print(df.dtypes)

CustomerID         object
First_Name         object
Last_Name          object
Phone_Number       object
Address            object
Paying Customer    object
Do_Not_Contact     object
Phone_Missing        bool
dtype: object


In [ ]:
summary = pd.DataFrame({
    'Metric': ['Row count', 'Null count', 'Duplicate count'],
    'Before': [21, 8, 1], # fill in your actual "before" numbers from the first .info()/duplicated() run
    'After': [len(df), df.isnull().sum().sum(), df.duplicated().sum()]
})
print(summary)

            Metric  Before  After
0        Row count      21     20
1       Null count       8      2
2  Duplicate count       1      0


In [ ]:
df.to_csv('cleaned_customer_data.csv', index=False)

### Missing Value Justification

After cleaning, 2 null values remain in the dataset both in `Phone_Number`.

These were intentionally **not imputed**, since there is no reliable way to guess
a missing phone number. Instead, a `Phone_Missing` boolean column was added to
flag these rows so they remain traceable in any downstream analysis.

All other missing values were resolved:
- `Last_Name` (1 missing) → filled with "Unknown"
- `Do_Not_Contact` (4 missing) → filled with "N" (assumed contactable unless explicitly opted out)

In [ ]:
df.to_csv('cleaned_customer_data.csv', index=False)

In [30]:
from google.colab import files
files.download('cleaned_customer_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>